<a href="https://colab.research.google.com/github/markajbell/BH/blob/main/ImportADSynth_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Imports

In [1]:
import pandas as pd
import json
import os
import glob

Set the Path to the Data location (imported form SharpHound)

# Set the Path
to the Data location (imported from SharpHound) and Read json files in folder and create a dictionary with the entire data.





In [2]:
#@title Download lab files
# from IPython.display import clear_output
# if not os.path.exists("/content/mylab"):
#   print("Downloading lab files...")
#  !wget https://raw.githubusercontent.com/markajbell/BH/refs/heads/main/mylab/20250316212118_computers.json
#  !wget https://raw.githubusercontent.com/markajbell/BH/refs/heads/main/mylab/20250316212118_containers.json
#  !wget https://raw.githubusercontent.com/markajbell/BH/refs/heads/main/mylab/20250316212118_domains.json
#  !wget https://raw.githubusercontent.com/markajbell/BH/refs/heads/main/mylab/20250316212118_gpos.json
#  !wget https://raw.githubusercontent.com/markajbell/BH/refs/heads/main/mylab/20250316212118_groups.json
#  !wget https://raw.githubusercontent.com/markajbell/BH/refs/heads/main/mylab/20250316212118_ous.json
#  !wget https://raw.githubusercontent.com/markajbell/BH/refs/heads/main/mylab/20250316212118_users.json
#   !wget https://raw.githubusercontent.com/markajbell/BH/refs/heads/main/mylab.zip
#   !unzip mylab.zip -d mylab
#   clear_output()
#   print("Done")
!wget https://raw.githubusercontent.com/markajbell/BH/refs/heads/main/adsync/secure.json
#!wget https://raw.githubusercontent.com/markajbell/BH/refs/heads/main/adsync/secure10k.json
# else:
#   print("Lab files already downloaded")
!mkdir = "/content/adsync"
!mv secure.json /content/adsync
#!mv secure10k.json /content/adsync

path = "/content/adsync"
data = {}
json_files = glob.glob(os.path.join(path, "*.json"))
for file in json_files:
    filename = os.path.basename(file).replace(".json", "")
    suf = filename.split("_")[-1]

    with open(file, 'r', encoding='utf-8') as f:
        content = json.load(f)

    if "data" in content:
        if suf in data:
            data[f"{suf}"].extend(content["data"])
            #print(f" The key 'data' was found in {f}.")
        else:
            data[f"{suf}"] = content["data"]
            #print(f" The key 'data' was found in {f}.")
    else:
        print(f"Warning: The key 'data' was not found in {f}.")

--2025-06-17 12:29:11--  https://raw.githubusercontent.com/markajbell/BH/refs/heads/main/adsync/secure.json
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1314149 (1.3M) [text/plain]
Saving to: ‘secure.json’

secure.json         100%[===================>]   1.25M  --.-KB/s    in 0.01s   

2025-06-17 12:29:11 (85.9 MB/s) - ‘secure.json’ saved [1314149/1314149]



In [3]:
suf

'secure'

Show the "content"

In [ ]:
content


In [ ]:
# prompt: from data['secure'] get the type column

# try:
#   secure_types = [item['type'] for item in data[suf]]
#   print(secure_types)
# except KeyError:
#   print("The key 'secure' was not found in the data dictionary.")
# except TypeError:
#   print("The value for 'secure' is not a list of dictionaries.")

In [6]:
# prompt: if the type column = "node" save that record into a pandas dataframe called nodes, if the type = "relationship' then save to a pandas dataframe called edges

import pandas as pd
df = pd.DataFrame(data[suf])
nodes = df[df['type'] == 'node'].copy()
edges = df[df['type'] == 'relationship'].copy()

In [8]:
# prompt: convert id in nodes to int

nodes['id'] = nodes['id'].astype(int)
nodes.head()

,id,labels,properties,type,start,end,label
0,0,"[Base, Domain]","{'domain': 'MARK.LOCAL', 'name': 'MARK.LOCAL',...",node,NaN,NaN,NaN
1,1,"[Base, OU]","{'domain': 'MARK.LOCAL', 'name': 'Admin@MARK.L...",node,NaN,NaN,NaN
2,2,"[Base, OU]","{'domain': 'MARK.LOCAL', 'name': 'Tier 1 Serve...",node,NaN,NaN,NaN
3,3,"[Base, OU]","{'domain': 'MARK.LOCAL', 'name': 'Tier 2@MARK....",node,NaN,NaN,NaN
4,4,"[Base, OU]","{'domain': 'MARK.LOCAL', 'name': 'T0 Admin@MAR...",node,NaN,NaN,NaN


In [9]:
nodes['id'].dtype

dtype('int64')

In [10]:
# prompt: show all rows

import pandas as pd
pd.set_option('display.max_rows', None)



In [11]:
# prompt: for every record in "nodes" create a l1, l2 and l3 column for the values in "nodes.labels" eg. for record 1 l1=Base, l2=OU and l3 would be null

import pandas as pd
def extract_labels(labels):
    # Ensure labels is a list
    if not isinstance(labels, list):
        return [None, None, None]

    l1 = labels[0] if len(labels) > 0 else None
    l2 = labels[1] if len(labels) > 1 else None
    l3 = labels[2] if len(labels) > 2 else None
    return [l1, l2, l3]

# Apply the function to create the new columns
nodes[['l1', 'l2', 'l3']] = nodes['labels'].apply(lambda x: pd.Series(extract_labels(x)))

nodes.head()

,id,labels,properties,type,start,end,label,l1,l2,l3
0,0,"[Base, Domain]","{'domain': 'MARK.LOCAL', 'name': 'MARK.LOCAL',...",node,NaN,NaN,NaN,Base,Domain,None
1,1,"[Base, OU]","{'domain': 'MARK.LOCAL', 'name': 'Admin@MARK.L...",node,NaN,NaN,NaN,Base,OU,None
2,2,"[Base, OU]","{'domain': 'MARK.LOCAL', 'name': 'Tier 1 Serve...",node,NaN,NaN,NaN,Base,OU,None
3,3,"[Base, OU]","{'domain': 'MARK.LOCAL', 'name': 'Tier 2@MARK....",node,NaN,NaN,NaN,Base,OU,None
4,4,"[Base, OU]","{'domain': 'MARK.LOCAL', 'name': 'T0 Admin@MAR...",node,NaN,NaN,NaN,Base,OU,None


In [12]:
# prompt: show me records where l3 ne "None"

nodes[nodes['l3'].notna()]

,id,labels,properties,type,start,end,label,l1,l2,l3
71,71,"[Base, Group, User]","{'domain': 'MARK.LOCAL', 'name': 'DMAPLE00091@...",node,NaN,NaN,NaN,Base,Group,User
83,83,"[Base, Group, User]","{'domain': 'MARK.LOCAL', 'name': 'MHONNETTE000...",node,NaN,NaN,NaN,Base,Group,User
396,396,"[Base, User, Compromised]","{'domain': 'MARK.LOCAL', 'objectid': 'S-1-5-21...",node,NaN,NaN,NaN,Base,User,Compromised
425,425,"[Base, User, Compromised]","{'domain': 'MARK.LOCAL', 'objectid': 'S-1-5-21...",node,NaN,NaN,NaN,Base,User,Compromised
428,428,"[Base, User, Compromised]","{'domain': 'MARK.LOCAL', 'objectid': 'S-1-5-21...",node,NaN,NaN,NaN,Base,User,Compromised
450,450,"[Base, User, Compromised]","{'domain': 'MARK.LOCAL', 'objectid': 'S-1-5-21...",node,NaN,NaN,NaN,Base,User,Compromised


In [13]:
# prompt: remove the following columns from nodes:
# start, end, label

nodes = nodes.drop(columns=['start', 'end', 'label', 'labels'])
nodes.head()

,id,properties,type,l1,l2,l3
0,0,"{'domain': 'MARK.LOCAL', 'name': 'MARK.LOCAL',...",node,Base,Domain,None
1,1,"{'domain': 'MARK.LOCAL', 'name': 'Admin@MARK.L...",node,Base,OU,None
2,2,"{'domain': 'MARK.LOCAL', 'name': 'Tier 1 Serve...",node,Base,OU,None
3,3,"{'domain': 'MARK.LOCAL', 'name': 'Tier 2@MARK....",node,Base,OU,None
4,4,"{'domain': 'MARK.LOCAL', 'name': 'T0 Admin@MAR...",node,Base,OU,None


In [ ]:
edges.tail()

,id,labels,properties,type,start,end,label
5266,r_3969,NaN,"{'isacl': True, 'isInherited': True, 'inherita...",relationship,"{'id': '65', 'labels': ['Base', 'Group']}","{'id': '3', 'labels': ['Base', 'OU']}",GenericAll
5267,r_3970,NaN,"{'isacl': True, 'isInherited': False, 'inherit...",relationship,"{'id': '66', 'labels': ['Base', 'Group']}","{'id': '1', 'labels': ['Base', 'OU']}",GenericAll
5268,r_3971,NaN,"{'isacl': True, 'isInherited': False, 'inherit...",relationship,"{'id': '66', 'labels': ['Base', 'Group']}","{'id': '3', 'labels': ['Base', 'OU']}",GenericAll
5269,r_3972,NaN,"{'isacl': True, 'isInherited': False, 'inherit...",relationship,"{'id': '60', 'labels': ['Base', 'Group']}","{'id': '101', 'labels': ['Base', 'OU']}",GenericAll
5270,r_3973,NaN,"{'isacl': True, 'isInherited': False, 'inherit...",relationship,"{'id': '66', 'labels': ['Base', 'Group']}","{'id': '101', 'labels': ['Base', 'OU']}",GenericAll


In [14]:
# prompt: remove the "labels" column from edges

edges = edges.drop(columns=['labels'])
edges.head()

,id,properties,type,start,end,label
1297,r_0,{},relationship,"{'id': '0', 'labels': ['Base', 'Domain']}","{'id': '1', 'labels': ['Base', 'OU']}",Contains
1298,r_1,{},relationship,"{'id': '0', 'labels': ['Base', 'Domain']}","{'id': '2', 'labels': ['Base', 'OU']}",Contains
1299,r_2,{},relationship,"{'id': '0', 'labels': ['Base', 'Domain']}","{'id': '3', 'labels': ['Base', 'OU']}",Contains
1300,r_3,{},relationship,"{'id': '1', 'labels': ['Base', 'OU']}","{'id': '4', 'labels': ['Base', 'OU']}",Contains
1301,r_4,{},relationship,"{'id': '1', 'labels': ['Base', 'OU']}","{'id': '5', 'labels': ['Base', 'OU']}",Contains


Expand "Properties" column in new columns.

Convert to Dict

In [ ]:
pd.set_option("display.max_rows", None, "display.max_columns", None)

#nodes

In [17]:
#edges = edges.drop(columns=['id'])
import numpy as np
edges['id'] = np.arange(len(edges)).astype('int64')
edges.head()
edges['id'].dtype

dtype('int64')

In [ ]:
# prompt: save the df_compuers as json to content

import json

# Assuming df_computers is already defined as in your provided code.

# Convert the DataFrame to a JSON string.
nodes_json = nodes.to_json(orient='records')

# Save the JSON string to a file named 'content.json'.
with open('nodes.json', 'w') as f:
  f.write(nodes_json)


In [50]:
# prompt: save the df_compuers as json to content

import json

# Assuming df_computers is already defined as in your provided code.

# Convert the DataFrame to a JSON string.
edges_json = edges.to_json(orient='records')

# Save the JSON string to a file named 'content.json'.
with open('edges.json', 'w') as f:
  f.write(edges_json)

In [19]:
# prompt: create a new column in nodes called weight

nodes['weight'] = 1

In [20]:
# prompt: create a list of all id from nodes where "highvalue" = true

# Before accessing 'Properties', check if the column exists in the DataFrame.
if 'properties' in nodes.columns:
    # Use .loc to avoid SettingWithCopyWarning and access the nested 'highvalue' key
    # We use a lambda function with .apply to safely access nested dictionary keys.
    high_value_node_ids = nodes[nodes['properties'].apply(lambda x: x.get('highvalue', False) == True)]['id'].tolist()
    print(high_value_node_ids)
else:
    print("Error: The 'Properties' column was not found in the nodes DataFrame.")

[0, 36, 38, 40, 59, 60, 65, 66, 72, 75, 90, 91, 92, 93, 94, 95, 96, 97, 98, 288, 289, 291, 293, 294, 295, 296, 297, 299, 301, 302, 303, 304, 305, 306, 307, 308, 309, 311, 312, 313, 314, 315, 318, 319, 320, 321, 322, 323, 324, 491, 501, 507, 508, 509, 511, 512, 514, 521, 523, 527, 530, 531, 532, 533, 534, 539, 544, 548, 550, 553, 557, 561, 562, 563, 564, 565, 567, 573, 575, 577, 585, 590, 597, 599, 600, 603, 607, 609, 610, 620, 623, 624, 625, 626, 631, 632, 637, 638, 639, 640, 645, 648, 649, 650, 651, 653, 654, 655, 656, 660, 661, 663, 665, 666, 668, 672, 673, 676, 677, 681, 684, 692, 693, 694]


In [21]:
len(high_value_node_ids)


124

In [22]:
# prompt: if node[id] = value in high_value_nodes_ids assign a weight of 20

for value in high_value_node_ids:
    nodes.loc[nodes['id'] == value, 'weight'] = 20


In [ ]:
pd.set_option("display.max_rows", None, "display.max_columns", None)
pd.set_option('display.max_colwidth', None)

#nodes.tail(50)

In [ ]:
# prompt: print all nodes where weight==20

#print(nodes[nodes['weight'] == 20])

In [ ]:
len(nodes)

1297

In [ ]:
# prompt: show me records where nodes['properties']['distinguishedname']  does not contain "T2", "T1" or "T0"

# Filter nodes where 'distinguishedname' in 'properties' does not contain "T2", "T1", or "T0"
filtered_nodes = nodes[
    nodes['properties'].apply(
        lambda x: 'distinguishedname' in x and
                  'T2' not in x['distinguishedname'] and
                  'T1' not in x['distinguishedname'] and
                  'T0' not in x['distinguishedname'] and
                  'Tier 0' not in x['distinguishedname'] and
                  'Tier 1' not in x['distinguishedname'] and
                  'Tier 2' not in x['distinguishedname'] and
                  'TIER 0' not in x['distinguishedname'] and
                  'TIER 1' not in x['distinguishedname'] and
                  'TIER 2' not in x['distinguishedname']
    )
]

filtered_nodes


,id,labels,properties,type,l1,l2,l3,weight
0,0,"[Base, Domain]","{'domain': 'MARK.LOCAL', 'name': 'MARK.LOCAL', 'highvalue': True, 'objectid': 'S-1-5-21-883232822-274137685-4173207997', 'distinguishedname': 'DC=MARK,DC=LOCAL', 'functionallevel': '2012', 'owned': False}",node,Base,Domain,None,20
1,1,"[Base, OU]","{'domain': 'MARK.LOCAL', 'name': 'Admin@MARK.LOCAL', 'objectid': 'S-1-5-21-883232822-274137685-4173207997-cdca22b1-d164-4ec1-8233-45c0ea9044ac', 'distinguishedname': 'OU=ADMIN,DC=MARK,DC=LOCAL', 'description': None, 'highvalue': False, 'blocksInheritance': False, 'owned': False}",node,Base,OU,None,1
19,19,"[Base, OU]","{'domain': 'MARK.LOCAL', 'name': 'Application@MARK.LOCAL', 'objectid': 'S-1-5-21-883232822-274137685-4173207997-c8fbc840-cc56-4337-aa2c-31984398cf7f', 'distinguishedname': 'OU=APPLICATION,DC=MARK,DC=LOCAL', 'description': None, 'highvalue': False, 'blocksInheritance': False, 'owned': False}",node,Base,OU,None,1
20,20,"[Base, OU]","{'domain': 'MARK.LOCAL', 'name': 'Print@MARK.LOCAL', 'objectid': 'S-1-5-21-883232822-274137685-4173207997-70355ec3-a132-4ec7-b4c7-f7cdfb011b59', 'distinguishedname': 'OU=PRINT,DC=MARK,DC=LOCAL', 'description': None, 'highvalue': False, 'blocksInheritance': False, 'owned': False}",node,Base,OU,None,1
21,21,"[Base, OU]","{'domain': 'MARK.LOCAL', 'name': 'Database@MARK.LOCAL', 'objectid': 'S-1-5-21-883232822-274137685-4173207997-bee8279d-bcc2-427c-827c-510159969430', 'distinguishedname': 'OU=DATABASE,DC=MARK,DC=LOCAL', 'description': None, 'highvalue': False, 'blocksInheritance': False, 'owned': False}",node,Base,OU,None,1
36,36,"[Base, Group]","{'domain': 'MARK.LOCAL', 'name': 'ADMINISTRATORS@MARK.LOCAL', 'objectid': 'MARK.LOCAL-S-1-5-32-544', 'highvalue': True, 'distinguishedname': 'CN=Administrators,CN=Builtin,DC=MARK,DC=LOCAL', 'description': 'Administrators have complete and unrestricted access to the computer/domain', 'admincount': True, 'owned': False}",node,Base,Group,None,20
37,37,"[Base, Group]","{'domain': 'MARK.LOCAL', 'name': 'REMOTE DESKTOP USERS@MARK.LOCAL', 'objectid': 'MARK.LOCAL-S-1-5-32-555', 'highvalue': False, 'distinguishedname': 'CN=Remote Desktop Users,CN=Builtin,DC=MARK,DC=LOCAL', 'description': 'Members in this group are granted the right to logon remotely', 'admincount': False, 'owned': False}",node,Base,Group,None,1
38,38,"[Base, Group]","{'domain': 'MARK.LOCAL', 'name': 'PRINT OPERATORS@MARK.LOCAL', 'objectid': 'MARK.LOCAL-S-1-5-32-550', 'highvalue': True, 'distinguishedname': 'CN=Print Operators,CN=Builtin,DC=MARK,DC=LOCAL', 'description': 'Members can administer printers installed on domain controllers', 'admincount': True, 'owned': False}",node,Base,Group,None,20
39,39,"[Base, Group]","{'domain': 'MARK.LOCAL', 'name': 'IIS_IUSRS@MARK.LOCAL', 'objectid': 'MARK.LOCAL-S-1-5-32-568', 'highvalue': False, 'distinguishedname': 'CN=IIS_IUSRS,CN=Builtin,DC=MARK,DC=LOCAL', 'description': 'Built-in group used by Internet Information Services.', 'admincount': False, 'owned': False}",node,Base,Group,None,1
40,40,"[Base, Group]","{'domain': 'MARK.LOCAL', 'name': 'BACKUP OPERATORS@MARK.LOCAL', 'objectid': 'MARK.LOCAL-S-1-5-32-551', 'highvalue': True, 'distinguishedname': 'CN=Backup Operators,CN=Builtin,DC=MARK,DC=LOCAL', 'description': 'Backup Operators can override security restrictions for the sole purpose of backing up or restoring files', 'admincount': True, 'owned': False}",node,Base,Group,None,20


In [ ]:
len(filtered_nodes)

68

In [ ]:
# prompt: show all column contents for nodes do not truncate columns

import pandas as pd
pd.set_option('display.max_colwidth', None)
#nodes

In [25]:
# prompt: compile a list of all id's from nodes where distinguishedname contains 'Tier 0' or T0. distinguishedname is a key within the properties column

tier_0_t0_ids = []
for index, row in nodes.iterrows():
    properties = row.get('properties')
    if properties and isinstance(properties, dict):
        distinguishedname = properties.get('distinguishedname')
        if distinguishedname and ('Tier 0' in distinguishedname or 'T0' in distinguishedname or 'TIER 0' in distinguishedname):
            tier_0_t0_ids.append(row['id'])

print("tier 0: ", tier_0_t0_ids)
len_tier0=len(tier_0_t0_ids)
print(len_tier0)

tier_1_t0_ids = []
for index, row in nodes.iterrows():
    properties = row.get('properties')
    if properties and isinstance(properties, dict):
        distinguishedname = properties.get('distinguishedname')
        if distinguishedname and ('Tier 1' in distinguishedname or 'T1' in distinguishedname or 'TIER 1' in distinguishedname):
            tier_1_t0_ids.append(row['id'])

print("tier 1: ",tier_1_t0_ids)
len_tier1=len(tier_1_t0_ids)
print(len_tier1)

tier_2_t0_ids = []
for index, row in nodes.iterrows():
    properties = row.get('properties')
    if properties and isinstance(properties, dict):
        distinguishedname = properties.get('distinguishedname')
        if distinguishedname and ('Tier 2' in distinguishedname or 'T2' in distinguishedname or 'TIER 2' in distinguishedname):
            tier_2_t0_ids.append(row['id'])

print("tier 2: ",tier_2_t0_ids)
len_tier2= len(tier_2_t0_ids)
print(len_tier2)
total = len_tier0 + len_tier1 + len_tier2
print("total: ",total)


tier 0:  [4, 7, 8, 9, 10, 295, 296, 301, 303, 305, 306, 307, 318, 324, 331, 341, 357, 392, 405, 444, 448, 457, 458, 467, 469, 481, 527, 530, 531, 532, 553, 607, 610, 625, 631, 632, 640, 645, 653, 663, 686, 692]
42
tier 1:  [5, 11, 12, 13, 14, 90, 91, 92, 93, 288, 291, 297, 302, 309, 315, 319, 323, 459, 460, 461, 467, 491, 508, 512, 514, 523, 534, 544, 550, 561, 563, 564, 575, 577, 585, 590, 597, 599, 600, 620, 623, 624, 626, 639, 648, 650, 651, 654, 665, 677, 681, 684, 687, 689, 690, 691, 693]
57
tier 2:  [6, 15, 16, 17, 18, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 71, 83, 94, 95, 96, 97, 289, 290, 292, 293, 294, 298, 299, 300, 301, 304, 308, 310, 311, 312, 313, 314, 316, 317, 318, 320, 321, 322, 325, 326, 327, 328, 329, 330, 331, 332, 333, 334, 335, 336, 337, 338, 339, 340, 341, 342, 343, 344, 345, 346, 347, 348, 349, 350, 351, 352, 353, 354, 355, 356, 357, 358, 359, 360, 361, 362, 363, 364, 365, 366, 367, 368, 369, 370, 371, 372, 373, 374, 375, 376, 377, 378, 379, 380,

In [26]:
# prompt: if tier_0_t0_ids isnot high_value_node_ids then append nodeid in a new dataframe called total_tier0

total_tier0 = []
for nodeid in tier_0_t0_ids:
  if nodeid not in high_value_node_ids:
    total_tier0.append(nodeid)

print("Nodes in tier_0_t0_ids that are not in high_value_node_ids:", total_tier0)

Nodes in tier_0_t0_ids that are not in high_value_node_ids: [4, 7, 8, 9, 10, 331, 341, 357, 392, 405, 444, 448, 457, 458, 467, 469, 481, 686]


In [27]:
# prompt: total_tier0 = total_tier0 and appended high_value_node_ids

total_tier0.extend(high_value_node_ids)
print("total_tier0:", total_tier0)


total_tier0: [4, 7, 8, 9, 10, 331, 341, 357, 392, 405, 444, 448, 457, 458, 467, 469, 481, 686, 0, 36, 38, 40, 59, 60, 65, 66, 72, 75, 90, 91, 92, 93, 94, 95, 96, 97, 98, 288, 289, 291, 293, 294, 295, 296, 297, 299, 301, 302, 303, 304, 305, 306, 307, 308, 309, 311, 312, 313, 314, 315, 318, 319, 320, 321, 322, 323, 324, 491, 501, 507, 508, 509, 511, 512, 514, 521, 523, 527, 530, 531, 532, 533, 534, 539, 544, 548, 550, 553, 557, 561, 562, 563, 564, 565, 567, 573, 575, 577, 585, 590, 597, 599, 600, 603, 607, 609, 610, 620, 623, 624, 625, 626, 631, 632, 637, 638, 639, 640, 645, 648, 649, 650, 651, 653, 654, 655, 656, 660, 661, 663, 665, 666, 668, 672, 673, 676, 677, 681, 684, 692, 693, 694]


In [28]:
len(total_tier0)

142

In [ ]:
len(nodes)

1297

In [ ]:
# prompt: how do i rewrite the following code to check for "TIER 0", "Tier 0" and "T0":
# tier0_node_ids = nodes[nodes['properties'].apply(lambda x: 'Tier 0' in x.get('distinguishedname', ''))]['id'].tolist()

tier0_node_ids = nodes[
    nodes['properties'].apply(
        lambda x: any(
            term in x.get('distinguishedname', '')
            for term in ['TIER 0', 'Tier 0', 'T0']
        )
    )
]['id'].tolist()


In [ ]:

tier0_node_ids

[4,
 7,
 8,
 9,
 10,
 295,
 296,
 301,
 303,
 305,
 306,
 307,
 318,
 324,
 331,
 341,
 357,
 392,
 405,
 444,
 448,
 457,
 458,
 467,
 469,
 481,
 527,
 530,
 531,
 532,
 553,
 607,
 610,
 625,
 631,
 632,
 640,
 645,
 653,
 663,
 686,
 692]

In [29]:
# 1/0
# tier0_node_ids = nodes[nodes['properties'].apply(lambda x: 'Tier 0' in x.get('distinguishedname', ''))]['id'].tolist()
# tier0_node_ids

tier0_node_ids = nodes[
    nodes['properties'].apply(
        lambda x: any(
            term in x.get('distinguishedname', '')
            for term in ['TIER 0', 'Tier 0', 'T0']
        )
    )
]['id'].tolist()


tier1_node_ids = nodes[
    nodes['properties'].apply(
        lambda x: any(
            term in x.get('distinguishedname', '')
            for term in ['TIER 1', 'Tier 1', 'T1']
        )
    )
]['id'].tolist()

tier2_node_ids = nodes[
    nodes['properties'].apply(
        lambda x: any(
            term in x.get('distinguishedname', '')
            for term in ['TIER 2', 'Tier 2', 'T2']
        )
    )
]['id'].tolist()

print(f"tier 0 ",len(tier0_node_ids))
# tier1_node_ids = nodes[nodes['properties'].apply(lambda x: 'Tier 1' in x.get('distinguishedname', ''))]['id'].tolist()
print(f"tier 1 ",len(tier1_node_ids))

# tier2_node_ids = nodes[nodes['properties'].apply(lambda x: 'Tier 2' in x.get('distinguishedname', ''))]['id'].tolist()
print(f"tier 2 ",len(tier2_node_ids))

tier 0  42
tier 1  58
tier 2  949


In [ ]:
len(nodes)

1297

In [ ]:
# T0="T0"
# T1="T1"
# T2="T2"


In [ ]:
# # prompt: produce a list of id's from nodes when nodes[properties][name] contains T0

# ids_with_T0 = nodes[nodes['properties'].apply(lambda x: T0 in x.get('name', ''))]['id'].tolist()
# #ids_with_T0
# ids_with_T1 = nodes[nodes['properties'].apply(lambda x: T1 in x.get('name', ''))]['id'].tolist()
# #ids_with_T1
# ids_with_T2 = nodes[nodes['properties'].apply(lambda x: T2 in x.get('name', ''))]['id'].tolist()
# #ids_with_T2

In [ ]:
# print(ids_with_T0)
# print(ids_with_T1)
# print(ids_with_T2)

# print(len(ids_with_T0))
# print(len(ids_with_T1))
# print(len(ids_with_T2))


In [ ]:

for value in ids_with_T0:
    nodes.loc[nodes['id'] == value, 'weight'] = 20

for value in ids_with_T1:
    nodes.loc[nodes['id'] == value, 'weight'] = 10

for value in ids_with_T2:
    nodes.loc[nodes['id'] == value, 'weight'] = 5

nodes.head(50)

,id,labels,properties,type,weight
0,6221,"[Base, Domain]","{'domain': 'TESTLAB.LOCALE', 'name': 'TESTLAB....",node,20
1,6222,"[Base, OU]","{'domain': 'TESTLAB.LOCALE', 'name': 'Admin@TE...",node,1
2,6223,"[Base, OU]","{'domain': 'TESTLAB.LOCALE', 'name': 'Tier 1 S...",node,1
3,6224,"[Base, OU]","{'domain': 'TESTLAB.LOCALE', 'name': 'Tier 2@T...",node,1
4,6225,"[Base, OU]","{'domain': 'TESTLAB.LOCALE', 'name': 'T0 Admin...",node,20
5,6226,"[Base, OU]","{'domain': 'TESTLAB.LOCALE', 'name': 'T1 Admin...",node,10
6,6227,"[Base, OU]","{'domain': 'TESTLAB.LOCALE', 'name': 'T2 Admin...",node,5
7,6228,"[Base, OU]","{'domain': 'TESTLAB.LOCALE', 'name': 'T0 Admin...",node,20
8,6229,"[Base, OU]","{'domain': 'TESTLAB.LOCALE', 'name': 'T0 Admin...",node,20
9,6230,"[Base, OU]","{'domain': 'TESTLAB.LOCALE', 'name': 'T0 Admin...",node,20


In [ ]:
# prompt: get a unique list of label from edges and save it to list called edges_labels

edges_labels = edges['label'].unique().tolist()
edges_labels

['Contains',
 'GpLink',
 'GenericAll',
 'MemberOf',
 'GenericWrite',
 'WriteDacl',
 'Owns',
 'WriteOwner',
 'GetChanges',
 'GetChangesAll',
 'AllExtendedRights',
 'AdminTo',
 'HasSession',
 'AllowedToDelegate',
 'ReadLAPSPassword',
 'CanRDP',
 'ExecuteDCOM',
 'AllowedToAct',
 'CanPSRemote',
 'ForceChangePassword',
 'AddMember',
 'AddSelf']

In [ ]:
# Apply the weights based on the 'label' column

def assign_weight(label):
    if label == "GenericAll":
        return 9
    elif label == "Owns":
        return 10
    elif label == "GenericWrite":
        return 2
    elif label == "AllExtendedRights":
        return 6
    elif label == "CanRDP":
        return 2
    elif label == "Contains":
        return 2
    elif label == "DCSync":
        return 8
    elif label == "WriteDacl":
        return 5
    elif label == "WriteOwner":
        return 7
    elif label == "AddKeyCredentialLink":
        return 6
    elif label == "AdminTo":
        return 8
    elif label == "MemberOf":
        return 1
    elif label == "CanPSRemote":
        return 2
    elif label == "ExecuteDCOM":
        return 2
    elif label == "GPLink":
        return 3
    elif label == "HasSession":
        return 9
    elif label == "ReadLAPSPassword":
        return 2
    elif label == "GetChanges":
        return 3
    elif label == "GetChangesAll":
        return 3
    elif label == "AddSelf":
        return 3
    elif label == "ForceChangePassword":
        return 3
    elif label == "AddMember":
        return 5
    elif label == "tier0":
        return 20
    else:
        return 1 # Default weight for other labels

edges['weight'] = edges['label'].apply(assign_weight)

# Display the updated edges_all DataFrame with the 'weight' column
print("\nEdges_all DataFrame with weight column:")
edges.head(50)
#clear_output()

In [31]:
# prompt: from edges['start'] get 'id' and from edges['end] get 'id'
# replace start with edges['start'][id'] and  end with edges['end']['id']

edges['start'] = edges['start'].apply(lambda x: x.get('id') if isinstance(x, dict) else x)
edges['end'] = edges['end'].apply(lambda x: x.get('id') if isinstance(x, dict) else x)
edges.head(30)

,id,properties,type,start,end,label,weight
1297,0,{},relationship,0,1,Contains,2
1298,1,{},relationship,0,2,Contains,2
1299,2,{},relationship,0,3,Contains,2
1300,3,{},relationship,1,4,Contains,2
1301,4,{},relationship,1,5,Contains,2
1302,5,{},relationship,1,6,Contains,2
1303,6,{},relationship,4,7,Contains,2
1304,7,{},relationship,4,8,Contains,2
1305,8,{},relationship,4,9,Contains,2
1306,9,{},relationship,4,10,Contains,2


In [32]:
len(edges)

3974

In [34]:
print("\nEdges_all DataFrame with weight column:")
edges.tail(50)


Edges_all DataFrame with weight column:


,id,properties,type,start,end,label,weight
5221,3924,"{'isacl': True, 'isInherited': True, 'inheritanceType': 'All'}",relationship,36,11,WriteOwner,7
5222,3925,"{'isacl': True, 'isInherited': True, 'inheritanceType': 'All'}",relationship,36,15,WriteOwner,7
5223,3926,"{'isacl': True, 'isInherited': True, 'inheritanceType': 'All'}",relationship,36,14,WriteOwner,7
5224,3927,"{'isacl': True, 'isInherited': True, 'inheritanceType': 'All'}",relationship,36,18,WriteOwner,7
5225,3928,"{'isacl': True, 'isInherited': True, 'inheritanceType': 'All'}",relationship,36,24,WriteOwner,7
5226,3929,"{'isacl': True, 'isInherited': False, 'inheritanceType': 'All'}",relationship,66,98,WriteOwner,7
5227,3930,"{'isacl': True, 'isInherited': False, 'inheritanceType': 'All'}",relationship,65,98,WriteOwner,7
5228,3931,"{'isacl': True, 'isInherited': False, 'inheritanceType': 'All'}",relationship,60,8,GenericAll,9
5229,3932,"{'isacl': True, 'isInherited': False, 'inheritanceType': 'All'}",relationship,60,12,GenericAll,9
5230,3933,"{'isacl': True, 'isInherited': False, 'inheritanceType': 'All'}",relationship,60,16,GenericAll,9


In [ ]:
# prompt: create a new dataframe called adj with the values of start['id'] and end['id'] from each record

# adj = edges[['start', 'end']].copy()
# adj['start'] = adj['start'].apply(lambda x: x['id']).astype(int)
# adj['end'] = adj['end'].apply(lambda x: x['id']).astype(int)
# adj.head()

In [ ]:
len(edges)
edges.head(20)

,properties,type,start,end,label,id,weight
1297,{},relationship,0,1,Contains,0,2
1298,{},relationship,0,2,Contains,1,2
1299,{},relationship,0,3,Contains,2,2
1300,{},relationship,1,4,Contains,3,2
1301,{},relationship,1,5,Contains,4,2
1302,{},relationship,1,6,Contains,5,2
1303,{},relationship,4,7,Contains,6,2
1304,{},relationship,4,8,Contains,7,2
1305,{},relationship,4,9,Contains,8,2
1306,{},relationship,4,10,Contains,9,2


In [ ]:
# prompt: from edges, convert start[id] into an integer

# edges['start'] = edges['start'].apply(lambda x: x['id']).astype(int)

In [ ]:
# edges['end'] = edges['end'].apply(lambda x: x['id']).astype(int)
# edges['start'] = edges['start'].apply(lambda x: x['id']).astype(int)

In [35]:

edges = edges.drop(columns=['properties', 'id'])

In [38]:
# prompt: create a unique id for each row, make sure it is of type int64
#edges = edges.drop(columns=['id'])
import numpy as np
edges['id'] = np.arange(len(edges)).astype('int64')
edges.head()
edges['id'].dtype

dtype('int64')

In [39]:

edges.keys()

Index(['type', 'start', 'end', 'label', 'weight', 'id'], dtype='object')

In [40]:
# prompt: filter where edges['id'] = tier2_node_ids and call the dataframe selected_ddges

selected_edges = edges[edges['id'].isin(tier2_node_ids)]

In [71]:
selected_nodes = nodes[nodes['id'].isin(tier2_node_ids)]

In [42]:
# prompt: set weight to 20 in selected_nodes

for value in selected_nodes['id']:
    selected_nodes.loc[selected_nodes['id'] == value, 'weight'] = 20


In [80]:
selected_nodes

,id,properties,type,l1,l2,l3,weight
3,3,"{'domain': 'MARK.LOCAL', 'name': 'Tier 2@MARK.LOCAL', 'objectid': 'S-1-5-21-883232822-274137685-4173207997-5de20ac8-e7ff-4f05-b597-268af392400e', 'distinguishedname': 'OU=TIER 2,DC=MARK,DC=LOCAL', 'description': None, 'highvalue': False, 'blocksInheritance': False, 'owned': False}",node,Base,OU,None,1
6,6,"{'domain': 'MARK.LOCAL', 'name': 'T2 Admin@MARK.LOCAL', 'objectid': 'S-1-5-21-883232822-274137685-4173207997-e4602f9d-12bf-4f9e-b667-f6ccd27fae90', 'distinguishedname': 'OU=T2 ADMIN,DC=MARK,DC=LOCAL', 'description': None, 'highvalue': False, 'blocksInheritance': False, 'owned': False}",node,Base,OU,None,1
15,15,"{'domain': 'MARK.LOCAL', 'name': 'T2 Admin Accounts@MARK.LOCAL', 'objectid': 'S-1-5-21-883232822-274137685-4173207997-61c17d81-4eb2-4829-94e7-c7ddd6ecd193', 'distinguishedname': 'OU=T2 ADMIN ACCOUNTS,DC=MARK,DC=LOCAL', 'description': None, 'highvalue': False, 'blocksInheritance': False, 'owned': False}",node,Base,OU,None,1
16,16,"{'domain': 'MARK.LOCAL', 'name': 'T2 Admin Devices@MARK.LOCAL', 'objectid': 'S-1-5-21-883232822-274137685-4173207997-5305b84a-5f72-4ee8-9726-81485bc11905', 'distinguishedname': 'OU=T2 ADMIN DEVICES,DC=MARK,DC=LOCAL', 'description': None, 'highvalue': False, 'blocksInheritance': False, 'owned': False}",node,Base,OU,None,1
17,17,"{'domain': 'MARK.LOCAL', 'name': 'T2 Admin Groups@MARK.LOCAL', 'objectid': 'S-1-5-21-883232822-274137685-4173207997-508809b7-63d4-4f53-8376-b31aa66c0af6', 'distinguishedname': 'OU=T2 ADMIN GROUPS,DC=MARK,DC=LOCAL', 'description': None, 'highvalue': False, 'blocksInheritance': False, 'owned': False}",node,Base,OU,None,1
18,18,"{'domain': 'MARK.LOCAL', 'name': 'T2 Admin Service Accounts@MARK.LOCAL', 'objectid': 'S-1-5-21-883232822-274137685-4173207997-e7679546-c753-4556-889a-67fd84f9d165', 'distinguishedname': 'OU=T2 ADMIN SERVICE ACCOUNTS,DC=MARK,DC=LOCAL', 'description': None, 'highvalue': False, 'blocksInheritance': False, 'owned': False}",node,Base,OU,None,1
22,22,"{'domain': 'MARK.LOCAL', 'name': 'T2 Groups@MARK.LOCAL', 'objectid': 'S-1-5-21-883232822-274137685-4173207997-f6d13355-d2f9-45cf-9ee6-fdc8230ae4e5', 'distinguishedname': 'OU=T2 GROUPS,DC=MARK,DC=LOCAL', 'description': None, 'highvalue': False, 'blocksInheritance': False, 'owned': False}",node,Base,OU,None,1
23,23,"{'domain': 'MARK.LOCAL', 'name': 'T2 Workstations@MARK.LOCAL', 'objectid': 'S-1-5-21-883232822-274137685-4173207997-66aadca5-ccbc-46d4-91e7-f74f2527bdb5', 'distinguishedname': 'OU=T2 WORKSTATIONS,DC=MARK,DC=LOCAL', 'description': None, 'highvalue': False, 'blocksInheritance': False, 'owned': False}",node,Base,OU,None,1
24,24,"{'domain': 'MARK.LOCAL', 'name': 'T2 User Accounts@MARK.LOCAL', 'objectid': 'S-1-5-21-883232822-274137685-4173207997-d4c0667d-16e8-4afd-8cf3-5f6a900f6913', 'distinguishedname': 'OU=T2 USER ACCOUNTS,DC=MARK,DC=LOCAL', 'description': None, 'highvalue': False, 'blocksInheritance': False, 'owned': False}",node,Base,OU,None,1
25,25,"{'domain': 'MARK.LOCAL', 'name': 'T2 Servers@MARK.LOCAL', 'objectid': 'S-1-5-21-883232822-274137685-4173207997-3f1588f2-5124-4111-8d63-caf4b57a87ca', 'distinguishedname': 'OU=T2 SERVERS,DC=MARK,DC=LOCAL', 'description': None, 'highvalue': False, 'blocksInheritance': False, 'owned': False}",node,Base,OU,None,1


In [ ]:
len(selected_edges)

949

In [ ]:
# import networkx as nx

# # Create a directed graph
# G = nx.DiGraph()

# # Add nodes with their weights
# for index, row in nodes.iterrows():
#     # Ensure node IDs are integers when adding to the graph
#     G.add_node(int(row['id']), weight=row['weight'])

# # Add edges with their weights
# for index, row in edges.iterrows():
#     # Ensure edge endpoints are integers when adding to the graph
#     G.add_edge(int(row['start']), int(row['end']), weight=row['weight'])

# # Compute shortest paths from each source node to all reachable nodes
# shortest_paths = {}
# for source in G.nodes():
#     # Use single_source_dijkstra_path to find paths from one source to all others
#     shortest_paths[source] = nx.single_source_dijkstra_path(G, source)

# # Filter out paths from a node to itself and store in a new dictionary
# shortest_paths_filtered = {
#     source: {
#         target: path
#         for target, path in paths.items()
#         if source != target
#     }
#     for source, paths in shortest_paths.items()
# }

# # Print the shortest paths (optional)
# # for source, paths in shortest_paths_filtered.items():
# #     print(f"Shortest paths from node {source}:")
# #     for target, path in paths.items():
# #         print(f"  To node {target}: {path}")

In [ ]:
tier2_node_ids


In [ ]:
# selected_edges['end'] = selected_edges['end'].apply(lambda x: x['id']).astype(int)
# selected_edges['start'] = selected_edges['start'].apply(lambda x: x['id']).astype(int)

In [47]:
selected_edges

,type,start,end,label,weight,id
1300,relationship,1,4,Contains,2,3
1303,relationship,4,7,Contains,2,6
1312,relationship,6,16,Contains,2,15
1313,relationship,6,17,Contains,2,16
1314,relationship,6,18,Contains,2,17
1315,relationship,2,19,Contains,2,18
1319,relationship,3,23,Contains,2,22
1320,relationship,3,24,Contains,2,23
1321,relationship,3,25,Contains,2,24
1322,relationship,24,26,Contains,2,25


In [74]:
import networkx as nx

# Create a directed graph
G = nx.DiGraph()

# Add nodes with their weights
for index, row in selected_nodes.iterrows():
    # Ensure node IDs are integers when adding to the graph
    G.add_node(int(row['id']), weight=row['weight'])

# Add edges with their weights
for index, row in edges.iterrows():
    # Ensure edge endpoints are integers when adding to the graph
    G.add_edge(row['start'], row['end'], weight=row['weight'])

# Compute shortest paths from each source node to all reachable nodes
shortest_paths = {}
for source in G.nodes():
    # Use single_source_dijkstra_path to find paths from one source to all others
    shortest_paths[source] = nx.single_source_dijkstra_path(G, source)

# Filter out paths from a node to itself and store in a new dictionary
shortest_paths_filtered = {
    source: {
        target: path
        for target, path in paths.items()
        if source != target
    }
    for source, paths in shortest_paths.items()
}

# Print the shortest paths (optional)
for source, paths in shortest_paths_filtered.items():
    print(f"Shortest paths from node {source}:")
    for target, path in paths.items():
        print(f"  To node {target}: {path}")

Streaming output truncated to the last 5000 lines.
  To node 831: ['870', '30', '569', '362', '780', '92', '711', '831']
  To node 832: ['870', '30', '569', '362', '780', '92', '711', '832']
  To node 833: ['870', '30', '569', '362', '780', '92', '711', '833']
  To node 835: ['870', '30', '569', '362', '780', '92', '711', '835']
  To node 847: ['870', '30', '569', '362', '780', '92', '711', '847']
  To node 848: ['870', '30', '569', '362', '780', '92', '711', '848']
  To node 851: ['870', '30', '569', '362', '780', '92', '711', '851']
  To node 857: ['870', '30', '569', '362', '780', '92', '711', '857']
  To node 864: ['870', '30', '569', '362', '780', '92', '711', '864']
  To node 868: ['870', '30', '569', '362', '780', '92', '711', '868']
  To node 869: ['870', '30', '569', '362', '780', '92', '711', '869']
  To node 871: ['870', '30', '569', '362', '780', '92', '711', '871']
  To node 872: ['870', '30', '569', '362', '780', '92', '711', '872']
  To node 873: ['870', '30', '569', '36

In [51]:
# prompt: save the df_compuers as json to content

import json

# Assuming df_computers is already defined as in your provided code.

# Convert the DataFrame to a JSON string.
selected_edges_json = selected_edges.to_json(orient='records')

# Save the JSON string to a file named 'content.json'.
with open('selected_edges.json', 'w') as f:
  f.write(selected_edges_json)

In [52]:
# prompt: save the df_compuers as json to content

import json

# Assuming df_computers is already defined as in your provided code.

# Convert the DataFrame to a JSON string.
selected_nodes_json = selected_nodes.to_json(orient='records')

# Save the JSON string to a file named 'content.json'.
with open('selected_nodes.json', 'w') as f:
  f.write(selected_nodes_json)

In [55]:
# prompt: export shortest_paths_filtered to json and save as shortest_paths_filtered_json

shortest_paths_filtered_json = json.dumps(shortest_paths_filtered, indent=2)

# Optionally, save the JSON string to a file
with open('shortest_paths_filtered.json', 'w') as f:
  f.write(shortest_paths_filtered_json)

In [86]:
# prompt: for each record in "shortest_paths_filtered" add up the weights and find the smallest and largest combined weights and the path for the largest weight

# Function to calculate the total weight of a path
def calculate_path_weight(graph, path):
    total_weight = 0
    for i in range(len(path) - 1):
        # Add node weight
        total_weight += graph.nodes[path[i]].get('weight', 0)
        # Add edge weight
        total_weight += graph[path[i]][path[i+1]].get('weight', 0)
    # Add the weight of the last node in the path
    if path:
        total_weight += graph.nodes[path[-1]].get('weight', 0)
    return total_weight

all_path_weights = {}

# Iterate through each source and its paths in shortest_paths_filtered
for source, paths_from_source in shortest_paths_filtered.items():
    all_path_weights[source] = {}
    # Iterate through each target and its path
    for target, path in paths_from_source.items():
        # Calculate the weight of the path using the original graph G
        weight = calculate_path_weight(G, path)
        all_path_weights[source][target] = weight

# Find the smallest and largest combined weights
min_weight = float('inf')
max_weight = float('-inf')
largest_weight_path = None
largest_weight_source = None
largest_weight_target = None

for source, paths_from_source in all_path_weights.items():
    for target, weight in paths_from_source.items():
        if weight < min_weight:
            min_weight_source = source
            min_weight_target = target
            min_weight_path = shortest_paths_filtered[source][target]
            min_weight = weight
        if weight > max_weight:
            max_weight = weight
            largest_weight_source = source
            largest_weight_target = target
            largest_weight_path = shortest_paths_filtered[source][target]

print(f"Smallest combined weight: {min_weight}")
print(f"Path for the smallest weight: {min_weight_path}")
print(f"Source for the smallest weight path: {min_weight_source}")
print(f"Target for the smallest weight path: {min_weight_target}")
print(f"Largest combined weight: {max_weight}")
print(f"Path for the largest weight: {largest_weight_path}")
print(f"Source for the largest weight path: {largest_weight_source}")
print(f"Target for the largest weight path: {largest_weight_target}")

Smallest combined weight: 1
Path for the smallest weight: ['64', '63']
Source for the smallest weight path: 64
Target for the smallest weight path: 63
Largest combined weight: 43
Path for the largest weight: ['660', '313', '94', '698', '710', '91', '26', '329', '700', '66', '286', '59', '19', '689']
Source for the largest weight path: 660
Target for the largest weight path: 689


In [91]:

print(nodes[nodes['id'] == 660])
print(nodes[nodes['id'] == 313])
print(nodes[nodes['id'] == 94])
print(nodes[nodes['id'] == 698])
print(nodes[nodes['id'] == 710])
print(nodes[nodes['id'] == 91])

      id  \
660  660   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  properties  \
660  {'name': 'PAW-00023@MARK.LOCAL', 'operatingsystem': 'Windows 10 Enterprise', 'enabled': True, 'haslaps': False, 'highvalue': True, 'lastlogontimestamp': -1, 'pwdlastset': 0, 'serviceprincipalnames': 'TERMSRV/PAW-00023@MARK.LOCAL,TERMSRV/PAW-00023@MARK.LOCAL,RestrictedKrbHost/PAW-00023@MARK,HOST/PA

In [78]:
selected_nodes

,id,properties,type,l1,l2,l3,weight
3,3,"{'domain': 'MARK.LOCAL', 'name': 'Tier 2@MARK.LOCAL', 'objectid': 'S-1-5-21-883232822-274137685-4173207997-5de20ac8-e7ff-4f05-b597-268af392400e', 'distinguishedname': 'OU=TIER 2,DC=MARK,DC=LOCAL', 'description': None, 'highvalue': False, 'blocksInheritance': False, 'owned': False}",node,Base,OU,None,1
6,6,"{'domain': 'MARK.LOCAL', 'name': 'T2 Admin@MARK.LOCAL', 'objectid': 'S-1-5-21-883232822-274137685-4173207997-e4602f9d-12bf-4f9e-b667-f6ccd27fae90', 'distinguishedname': 'OU=T2 ADMIN,DC=MARK,DC=LOCAL', 'description': None, 'highvalue': False, 'blocksInheritance': False, 'owned': False}",node,Base,OU,None,1
15,15,"{'domain': 'MARK.LOCAL', 'name': 'T2 Admin Accounts@MARK.LOCAL', 'objectid': 'S-1-5-21-883232822-274137685-4173207997-61c17d81-4eb2-4829-94e7-c7ddd6ecd193', 'distinguishedname': 'OU=T2 ADMIN ACCOUNTS,DC=MARK,DC=LOCAL', 'description': None, 'highvalue': False, 'blocksInheritance': False, 'owned': False}",node,Base,OU,None,1
16,16,"{'domain': 'MARK.LOCAL', 'name': 'T2 Admin Devices@MARK.LOCAL', 'objectid': 'S-1-5-21-883232822-274137685-4173207997-5305b84a-5f72-4ee8-9726-81485bc11905', 'distinguishedname': 'OU=T2 ADMIN DEVICES,DC=MARK,DC=LOCAL', 'description': None, 'highvalue': False, 'blocksInheritance': False, 'owned': False}",node,Base,OU,None,1
17,17,"{'domain': 'MARK.LOCAL', 'name': 'T2 Admin Groups@MARK.LOCAL', 'objectid': 'S-1-5-21-883232822-274137685-4173207997-508809b7-63d4-4f53-8376-b31aa66c0af6', 'distinguishedname': 'OU=T2 ADMIN GROUPS,DC=MARK,DC=LOCAL', 'description': None, 'highvalue': False, 'blocksInheritance': False, 'owned': False}",node,Base,OU,None,1
18,18,"{'domain': 'MARK.LOCAL', 'name': 'T2 Admin Service Accounts@MARK.LOCAL', 'objectid': 'S-1-5-21-883232822-274137685-4173207997-e7679546-c753-4556-889a-67fd84f9d165', 'distinguishedname': 'OU=T2 ADMIN SERVICE ACCOUNTS,DC=MARK,DC=LOCAL', 'description': None, 'highvalue': False, 'blocksInheritance': False, 'owned': False}",node,Base,OU,None,1
22,22,"{'domain': 'MARK.LOCAL', 'name': 'T2 Groups@MARK.LOCAL', 'objectid': 'S-1-5-21-883232822-274137685-4173207997-f6d13355-d2f9-45cf-9ee6-fdc8230ae4e5', 'distinguishedname': 'OU=T2 GROUPS,DC=MARK,DC=LOCAL', 'description': None, 'highvalue': False, 'blocksInheritance': False, 'owned': False}",node,Base,OU,None,1
23,23,"{'domain': 'MARK.LOCAL', 'name': 'T2 Workstations@MARK.LOCAL', 'objectid': 'S-1-5-21-883232822-274137685-4173207997-66aadca5-ccbc-46d4-91e7-f74f2527bdb5', 'distinguishedname': 'OU=T2 WORKSTATIONS,DC=MARK,DC=LOCAL', 'description': None, 'highvalue': False, 'blocksInheritance': False, 'owned': False}",node,Base,OU,None,1
24,24,"{'domain': 'MARK.LOCAL', 'name': 'T2 User Accounts@MARK.LOCAL', 'objectid': 'S-1-5-21-883232822-274137685-4173207997-d4c0667d-16e8-4afd-8cf3-5f6a900f6913', 'distinguishedname': 'OU=T2 USER ACCOUNTS,DC=MARK,DC=LOCAL', 'description': None, 'highvalue': False, 'blocksInheritance': False, 'owned': False}",node,Base,OU,None,1
25,25,"{'domain': 'MARK.LOCAL', 'name': 'T2 Servers@MARK.LOCAL', 'objectid': 'S-1-5-21-883232822-274137685-4173207997-3f1588f2-5124-4111-8d63-caf4b57a87ca', 'distinguishedname': 'OU=T2 SERVERS,DC=MARK,DC=LOCAL', 'description': None, 'highvalue': False, 'blocksInheritance': False, 'owned': False}",node,Base,OU,None,1


In [49]:
selected_nodes

,id,properties,type,l1,l2,l3,weight
0,0,"{'domain': 'MARK.LOCAL', 'name': 'MARK.LOCAL', 'highvalue': True, 'objectid': 'S-1-5-21-883232822-274137685-4173207997', 'distinguishedname': 'DC=MARK,DC=LOCAL', 'functionallevel': '2012', 'owned': False}",node,Base,Domain,None,20
4,4,"{'domain': 'MARK.LOCAL', 'name': 'T0 Admin@MARK.LOCAL', 'objectid': 'S-1-5-21-883232822-274137685-4173207997-f249f685-d5e0-4dad-b5b5-7f723c41ca1e', 'distinguishedname': 'OU=T0 ADMIN,DC=MARK,DC=LOCAL', 'description': None, 'highvalue': False, 'blocksInheritance': False, 'owned': False}",node,Base,OU,None,20
7,7,"{'domain': 'MARK.LOCAL', 'name': 'T0 Admin Accounts@MARK.LOCAL', 'objectid': 'S-1-5-21-883232822-274137685-4173207997-c0fb853d-9b96-449b-b25c-4d20a1c02e5f', 'distinguishedname': 'OU=T0 ADMIN ACCOUNTS,DC=MARK,DC=LOCAL', 'description': None, 'highvalue': False, 'blocksInheritance': False, 'owned': False}",node,Base,OU,None,20
8,8,"{'domain': 'MARK.LOCAL', 'name': 'T0 Admin Devices@MARK.LOCAL', 'objectid': 'S-1-5-21-883232822-274137685-4173207997-b9fd1aed-b3e6-4d53-9242-b85ff389822f', 'distinguishedname': 'OU=T0 ADMIN DEVICES,DC=MARK,DC=LOCAL', 'description': None, 'highvalue': False, 'blocksInheritance': False, 'owned': False}",node,Base,OU,None,20
9,9,"{'domain': 'MARK.LOCAL', 'name': 'T0 Admin Groups@MARK.LOCAL', 'objectid': 'S-1-5-21-883232822-274137685-4173207997-cd13b9d6-76ca-4b5c-82bd-2d86bef8a375', 'distinguishedname': 'OU=T0 ADMIN GROUPS,DC=MARK,DC=LOCAL', 'description': None, 'highvalue': False, 'blocksInheritance': False, 'owned': False}",node,Base,OU,None,20
10,10,"{'domain': 'MARK.LOCAL', 'name': 'T0 Admin Service Accounts@MARK.LOCAL', 'objectid': 'S-1-5-21-883232822-274137685-4173207997-58cd5b4f-82db-4cd1-a3f4-08661c9c53d2', 'distinguishedname': 'OU=T0 ADMIN SERVICE ACCOUNTS,DC=MARK,DC=LOCAL', 'description': None, 'highvalue': False, 'blocksInheritance': False, 'owned': False}",node,Base,OU,None,20
36,36,"{'domain': 'MARK.LOCAL', 'name': 'ADMINISTRATORS@MARK.LOCAL', 'objectid': 'MARK.LOCAL-S-1-5-32-544', 'highvalue': True, 'distinguishedname': 'CN=Administrators,CN=Builtin,DC=MARK,DC=LOCAL', 'description': 'Administrators have complete and unrestricted access to the computer/domain', 'admincount': True, 'owned': False}",node,Base,Group,None,20
38,38,"{'domain': 'MARK.LOCAL', 'name': 'PRINT OPERATORS@MARK.LOCAL', 'objectid': 'MARK.LOCAL-S-1-5-32-550', 'highvalue': True, 'distinguishedname': 'CN=Print Operators,CN=Builtin,DC=MARK,DC=LOCAL', 'description': 'Members can administer printers installed on domain controllers', 'admincount': True, 'owned': False}",node,Base,Group,None,20
40,40,"{'domain': 'MARK.LOCAL', 'name': 'BACKUP OPERATORS@MARK.LOCAL', 'objectid': 'MARK.LOCAL-S-1-5-32-551', 'highvalue': True, 'distinguishedname': 'CN=Backup Operators,CN=Builtin,DC=MARK,DC=LOCAL', 'description': 'Backup Operators can override security restrictions for the sole purpose of backing up or restoring files', 'admincount': True, 'owned': False}",node,Base,Group,None,20
59,59,"{'domain': 'MARK.LOCAL', 'name': 'SERVER OPERATORS@MARK.LOCAL', 'objectid': 'MARK.LOCAL-S-1-5-32-549', 'highvalue': True, 'distinguishedname': 'CN=Server Operators,CN=Builtin,DC=MARK,DC=LOCAL', 'description': 'Members can administer domain servers', 'admincount': True, 'owned': False}",node,Base,Group,None,20


In [ ]:
pd.set_option("display.max_rows", None, "display.max_columns", None)
shortest_paths

{0: {0: [0],
  285: [0, 285],
  287: [0, 287],
  49: [0, 285, 49],
  74: [0, 285, 74],
  63: [0, 287, 63],
  86: [0, 285, 74, 86],
  87: [0, 285, 74, 87],
  50: [0, 285, 74, 87, 50]},
 4: {4: [4], 7: [4, 7]},
 7: {7: [7]},
 8: {8: [8],
  686: [8, 686],
  527: [8, 527],
  530: [8, 530],
  531: [8, 531],
  532: [8, 532],
  553: [8, 553],
  607: [8, 607],
  610: [8, 610],
  625: [8, 625],
  631: [8, 631],
  632: [8, 632],
  640: [8, 640],
  645: [8, 645],
  653: [8, 653],
  663: [8, 663]},
 9: {9: [9],
  65: [9, 65],
  284: [9, 65, 36, 284],
  285: [9, 65, 36, 285],
  286: [9, 65, 286],
  287: [9, 65, 36, 287],
  63: [9, 65, 64, 63],
  36: [9, 65, 36],
  37: [9, 65, 37],
  38: [9, 65, 286, 38],
  39: [9, 65, 39],
  40: [9, 65, 40],
  41: [9, 65, 36, 41],
  42: [9, 65, 36, 42],
  43: [9, 65, 43],
  45: [9, 65, 45],
  48: [9, 65, 36, 48],
  50: [9, 65, 36, 50],
  51: [9, 65, 36, 51],
  52: [9, 65, 52],
  53: [9, 65, 36, 53],
  56: [9, 65, 36, 56],
  57: [9, 65, 36, 57],
  58: [9, 65, 36, 58

In [ ]:
shortest_paths_filtered

{0: {285: [0, 285],
  287: [0, 287],
  49: [0, 285, 49],
  74: [0, 285, 74],
  63: [0, 287, 63],
  86: [0, 285, 74, 86],
  87: [0, 285, 74, 87],
  50: [0, 285, 74, 87, 50]},
 4: {7: [4, 7]},
 7: {},
 8: {686: [8, 686],
  527: [8, 527],
  530: [8, 530],
  531: [8, 531],
  532: [8, 532],
  553: [8, 553],
  607: [8, 607],
  610: [8, 610],
  625: [8, 625],
  631: [8, 631],
  632: [8, 632],
  640: [8, 640],
  645: [8, 645],
  653: [8, 653],
  663: [8, 663]},
 9: {65: [9, 65],
  284: [9, 65, 36, 284],
  285: [9, 65, 36, 285],
  286: [9, 65, 286],
  287: [9, 65, 36, 287],
  63: [9, 65, 64, 63],
  36: [9, 65, 36],
  37: [9, 65, 37],
  38: [9, 65, 286, 38],
  39: [9, 65, 39],
  40: [9, 65, 40],
  41: [9, 65, 36, 41],
  42: [9, 65, 36, 42],
  43: [9, 65, 43],
  45: [9, 65, 45],
  48: [9, 65, 36, 48],
  50: [9, 65, 36, 50],
  51: [9, 65, 36, 51],
  52: [9, 65, 52],
  53: [9, 65, 36, 53],
  56: [9, 65, 36, 56],
  57: [9, 65, 36, 57],
  58: [9, 65, 36, 58],
  59: [9, 65, 286, 59],
  60: [9, 65, 60]

In [ ]:
# prompt: can you display the combined weight from the shortest_paths and display the path with the max weight

# Function to calculate the total weight of a path
def calculate_path_weight(graph, path):
    total_weight = 0
    # Add the weight of the starting node
    if path:
        total_weight += graph.nodes[path[0]].get('weight', 0) # Get node weight
    # Add the weight of the edges and the weights of subsequent nodes
    for i in range(len(path) - 1):
        u, v = path[i], path[i+1]
        # Add edge weight
        total_weight += graph.edges[u, v].get('weight', 0) # Get edge weight
        # Add weight of the next node
        total_weight += graph.nodes[v].get('weight', 0) # Get node weight
    return total_weight

max_weight = -1
max_path = None

# Iterate through the filtered shortest paths and calculate total weights
for source, paths in shortest_paths_filtered.items():
    for target, path in paths.items():
        current_weight = calculate_path_weight(G, path)
#        print(f"Path from {source} to {target}: {path}, Combined Weight: {current_weight}")
        if current_weight > max_weight:
            max_weight = current_weight
            max_path = path

print(f"\nPath with the maximum combined weight: {max_path}")
print(f"Maximum combined weight: {max_weight}")


Path with the maximum combined weight: [9, 65, 36, 0]
Maximum combined weight: 87.0


In [ ]:
max_path
max_path = pd.DataFrame(max_path)
max_path.head()


,0
0,9
1,65
2,36
3,0


In [ ]:
rename = {0: 'id'}
max_path = max_path.rename(columns=rename)
max_path

,id
0,9
1,65
2,36
3,0


In [ ]:
# prompt: get label from nodes where id=max_path['id']

# Assuming max_path is a DataFrame with an 'id' column representing the node IDs in the max path
# and nodes is a DataFrame with 'id' and 'properties' columns.

# Get the list of node IDs from the max_path DataFrame
max_path_node_ids = max_path['id'].tolist()

# Filter the nodes DataFrame to get only the nodes that are in the max_path
nodes_in_max_path = nodes[nodes['id'].isin(max_path_node_ids)].copy()

# Now, extract the 'name' property from the 'properties' column for these nodes
# Use .loc to avoid SettingWithCopyWarning
# We use a lambda function with .apply to safely access nested dictionary keys.
max_path_node_labels = nodes_in_max_path.loc[:, 'properties'].apply(lambda x: x.get('name', ''))

print("Labels of nodes in the max path:")
print(max_path_node_labels.tolist())


Labels of nodes in the max path:
['MARK.LOCAL', 'T0 Admin Groups@MARK.LOCAL', 'ADMINISTRATORS@MARK.LOCAL', 'ENTERPRISE ADMINS@MARK.LOCAL']


In [ ]:
max_path

,id
0,9
1,65
2,36
3,0


ignore from here


In [ ]:
vertices = nodes['id'].tolist()
vertices


[6221,
 6222,
 6223,
 6224,
 6225,
 6226,
 6227,
 6228,
 6229,
 6230,
 6231,
 6232,
 6233,
 6234,
 6235,
 6236,
 6237,
 6238,
 6239,
 6240,
 6241,
 6242,
 6243,
 6244,
 6245,
 6246,
 6247,
 6248,
 6249,
 6250,
 6251,
 6252,
 6253,
 6254,
 6255,
 6256,
 6257,
 6258,
 6259,
 6260,
 6261,
 6262,
 6263,
 6264,
 6265,
 6266,
 6267,
 6268,
 6269,
 6270,
 6271,
 6272,
 6273,
 6274,
 6275,
 6276,
 6277,
 6278,
 6279,
 6280,
 6281,
 6282,
 6283,
 6284,
 6285,
 6286,
 6287,
 6288,
 6289,
 6290,
 6291,
 6292,
 6293,
 6294,
 6295,
 6296,
 6297,
 6298,
 6299,
 6300,
 6301,
 6302,
 6303,
 6304,
 6305,
 6306,
 6307,
 6308,
 6309,
 6310,
 6311,
 6312,
 6313,
 6314,
 6315,
 6316,
 6317,
 6318,
 6319,
 6320,
 6321,
 6322,
 6323,
 6324,
 6325,
 6326,
 6327,
 6328,
 6329,
 6330,
 6331,
 6332,
 6333,
 6334,
 6335,
 6336,
 6337,
 6338,
 6339,
 6340,
 6341,
 6342,
 6343,
 6344,
 6345,
 6346,
 6347,
 6348,
 6349,
 6350,
 6351,
 6352,
 6353,
 6354,
 6355,
 6356,
 6357,
 6358,
 6359,
 6360,
 6361,
 6362,
 6363,

In [ ]:
adj_list=adj.values.tolist()
len(adj_list)

48321

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

def createAdjacencyMatrix(vertices,edges):
  noofvertices=len(vertices)
  adjM=[]
  while(len(adjM)<noofvertices):
    temp=[]
    for i in range(noofvertices):
      temp.append(0)
    adjM.append(temp)
  for edge in edges:
    i=edge[0]
    j=edge[1]
    if i>=noofvertices or j>=noofvertices or i<0 or j<0:
      print(f"Not a Proper Input in Edge {i},{j}")
    else:
      adjM[i][j]=1
      adjM[j][i]=1
#  G=nx.Graph()
#  G.add_edges_from(edges)
#  nx.draw_networkx(G)
#  plt.show()
  return adjM
vertices = vertices
edges = adj_list
adjM=createAdjacencyMatrix(vertices,edges)


Not a Proper Input in Edge 10075,6298
Not a Proper Input in Edge 10077,6298
Not a Proper Input in Edge 10078,6298
Not a Proper Input in Edge 10080,6298
Not a Proper Input in Edge 10081,6298
Not a Proper Input in Edge 10082,6298
Not a Proper Input in Edge 10083,6298
Not a Proper Input in Edge 10085,6298
Not a Proper Input in Edge 10087,6298
Not a Proper Input in Edge 10088,6298
Not a Proper Input in Edge 10089,6298
Not a Proper Input in Edge 10090,6298
Not a Proper Input in Edge 10094,6298
Not a Proper Input in Edge 10100,6298
Not a Proper Input in Edge 10101,6298
Not a Proper Input in Edge 10103,6298
Not a Proper Input in Edge 10104,6298
Not a Proper Input in Edge 10105,6298
Not a Proper Input in Edge 10106,6298
Not a Proper Input in Edge 10107,6298
Not a Proper Input in Edge 10108,6298
Not a Proper Input in Edge 10110,6298
Not a Proper Input in Edge 10111,6298
Not a Proper Input in Edge 10114,6298
Not a Proper Input in Edge 10116,6298
Not a Proper Input in Edge 10117,6298
Not a Proper

In [ ]:
#adjM.shape
import numpy as np
len(adjM)
arr = np.array(adjM)

newarr = arr.reshape(10075, 10075)

print(newarr)

[[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]


In [ ]:
def create_adjacency_matrix(V, edges):
    # Initialize an empty V x V matrix with all zeros
    matrix = [[0] * V for _ in range(V)]

    # Populate the matrix based on the edges
    for edge in edges:
        u, v = edge
        matrix[u][v] = 1
#       matrix[v][u] = 1  # Undirected graph

    return matrix
V1 = len(adj)
edges1 = adj[['start', 'end']].values.tolist()
adj_matrix1 = create_adjacency_matrix(V1, edges1)
for row in adj_matrix1:
    print(row)
print()

In [ ]:
type(adj)

pandas.core.frame.DataFrame